In [ ]:
import os
try:
    import pandas as pd
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "openpyxl"])
    import pandas as pd
import json
import sqlite3
from unstructured.partition.auto import partition

def process_data_folder(folder_path):
    """
    Scans a folder and reads every file based on its extension.
    Returns a dictionary where the key is the filename and the value is the extracted data.
    """
    print(f"Scanning folder: {folder_path}\n" + "="*30)
    all_data = {} 
    
    # os.listdir gets everything inside the folder
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        # Skip sub-folders, we only want to read files
        if not os.path.isfile(file_path):
            continue
            
        # Split the filename to isolate the extension (e.g., '.pdf')
        _, file_extension = os.path.splitext(filename)
        file_extension = file_extension.lower() # ensure it is lowercase
        
        print(f"Processing: {filename}...")
        
        try:
            # 1. Structured Data: Excel
            if file_extension in ['.xlsx', '.xls']:
                df = pd.read_excel(file_path, engine='openpyxl')
                all_data[filename] = df
                print(f"  -> Success: Loaded Excel with {len(df)} rows.")
                
            # 2. Structured Data: JSON
            elif file_extension == '.json':
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                all_data[filename] = data
                print(f"  -> Success: Loaded JSON file.")
                
            # 3. Structured Data: SQLite Database
            elif file_extension in ['.db', '.sqlite']:
                conn = sqlite3.connect(file_path)
                # Query to find the names of all tables in the database
                tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
                all_data[filename] = {"connection": conn, "tables": tables['name'].tolist()}
                print(f"  -> Success: Connected to DB. Found tables: {tables['name'].tolist()}")
                
            # 4. Unstructured Data: PDF, Word, Text
            elif file_extension in ['.pdf', '.docx', '.doc', '.txt']:
                # 'partition' automatically detects the format.
                # We keep 'hi_res' so it still catches Arabic text and tables correctly!
                elements = partition(filename=file_path, strategy="hi_res")
                all_data[filename] = elements
                print(f"  -> Success: Extracted {len(elements)} structural elements.")
                
            else:
                print(f"  -> Skipped: Unsupported file type ({file_extension}).")
                
        except Exception as e:
            print(f"  -> Error loading {filename}: {e}")
            
    return all_data

# Run the function on our new 'data' folder
# Make sure your folder is named exactly 'data'
corpus = process_data_folder("./data")

print("\n" + "="*30)
print(f"Total files successfully loaded: {len(corpus)}")

ModuleNotFoundError: No module named 'pandas'